# Instalacja Ollama — jednorazowy setup

Ten notebook sprawdzi Twój sprzęt, zainstaluje Ollama i pobierze odpowiedni model LLM.

**Uruchom wszystkie komórki po kolei (Shift+Enter).**

In [1]:
# === Krok 1: Sprawdzenie sprzętu ===
import platform
import subprocess
import sys
import os
import requests

def get_ram_gb():
    system = platform.system()
    try:
        if system == 'Darwin':  # macOS
            out = subprocess.check_output(['sysctl', '-n', 'hw.memsize']).decode().strip()
            return int(out) / (1024 ** 3)
        elif system == 'Linux':
            with open('/proc/meminfo') as f:
                for line in f:
                    if 'MemTotal' in line:
                        return int(line.split()[1]) / (1024 ** 2)
        elif system == 'Windows':
            out = subprocess.check_output(
                ['wmic', 'computersystem', 'get', 'TotalPhysicalMemory'],
                shell=True
            ).decode()
            return int(out.strip().split()[-1]) / (1024 ** 3)
    except Exception as e:
        print(f'Nie udało się odczytać RAM: {e}')
    return None

def pick_model_for_ram(ram_gb):
    if ram_gb >= 28:
        return 'qwen3.5:27b', 'Świetnie! Duże modele nie są problemem.'
    elif ram_gb >= 14:
        return 'qwen3.5:9b', 'Dobre — pójdzie sprawny 9B model.'
    elif ram_gb >= 10:
        return 'qwen3.5:4b', 'OK — solidny 4B model.'
    elif ram_gb >= 6:
        return 'gemma4:e2b', 'Minimum — mały ale sprawny model.'
    else:
        return None, 'Za mało RAM — skorzystaj z serwera prowadzącego.'

system = platform.system()
ram_gb = get_ram_gb()
model, comment = pick_model_for_ram(ram_gb) if ram_gb else (None, 'Nie udało się odczytać RAM.')

print(f'System:       {system}')
print(f'RAM:          {ram_gb:.1f} GB' if ram_gb else 'RAM:          nieznany')
print(f'Rekomendowany model: {model or "brak"}')
print(f'Ocena:        {comment}')

System:       Darwin
RAM:          32.0 GB
Rekomendowany model: qwen3:14b
Ocena:        Świetnie! Duże modele nie są problemem.


In [2]:
# === Krok 2: Sprawdzenie czy Ollama już działa ===

def ollama_running():
    try:
        r = requests.get('http://localhost:11434/api/tags', timeout=3)
        return r.status_code == 200
    except:
        return False

if ollama_running():
    print('Ollama już działa! Pomijam instalację.')
    OLLAMA_INSTALLED = True
else:
    print('Ollama nie znaleziona — przejdź do kolejnej komórki.')
    OLLAMA_INSTALLED = False

Ollama już działa! Pomijam instalację.


In [3]:
# === Krok 3: Instalacja Ollama (pomiń jeśli już zainstalowana) ===

if OLLAMA_INSTALLED:
    print('Ollama już zainstalowana — nic do roboty.')

elif model is None:
    print('Za mało RAM — Ollama nie zostanie zainstalowana.')
    print('Na zajęciach skorzystaj z serwera prowadzącego.')

elif system in ('Darwin', 'Linux'):
    print('Instaluję Ollama (macOS/Linux)...')
    print('Może pojawić się prośba o hasło administratora.\n')
    result = subprocess.run(
        'curl -fsSL https://ollama.com/install.sh | sh',
        shell=True, capture_output=False
    )
    if result.returncode == 0:
        print('\nOllama zainstalowana!')
        OLLAMA_INSTALLED = True
    else:
        print('\nBłąd instalacji — spróbuj ręcznie: https://ollama.com/download')

elif system == 'Windows':
    print('Na Windowsie automatyczna instalacja jest utrudniona.')
    print('Pobierz instalator ręcznie: https://ollama.com/download')
    print('Po zainstalowaniu wróć i uruchom tę komórkę ponownie.')

Ollama już zainstalowana — nic do roboty.


In [5]:
# === Krok 4: Pobranie modelu ===
# To może potrwać kilka minut — model waży od 2 do 9 GB

if not model:
    print('Brak rekomendowanego modelu — nic do pobrania.')

elif not ollama_running():
    print('Ollama nie działa — uruchom ją i wróć tutaj.')
    if system == 'Darwin':
        print('Na Macu: otwórz aplikację Ollama z folderu Applications.')
    elif system == 'Linux':
        print('Na Linuksie: ollama serve (w osobnym terminalu)')

else:
    # Sprawdź czy model już jest pobrany
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    installed = [m['name'] for m in r.json().get('models', [])]
    already = any(model.split(':')[0] in m for m in installed)

    if already:
        print(f'Model {model} już jest pobrany!')
    else:
        print(f'Pobieram model {model}...')
        print('(może potrwać kilka minut — zależy od łącza)\n')
        result = subprocess.run(
            ['ollama', 'pull', model],
            capture_output=False
        )
        if result.returncode == 0:
            print(f'\nModel {model} gotowy!')
        else:
            print(f'\nBłąd pobierania modelu.')

Pobieram model qwen3:14b...
(może potrwać kilka minut — zależy od łącza)



pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest 
pulling a8cc1361f314:   0% ▕                  ▏ 3.1 MB/9.3 GB                  pulling manifest 
pulling a8cc1361f314:   0% ▕                  ▏ 8.5 MB/9.3 GB                  pulling manifest 
pulling a8cc1361f314:   0% ▕                  ▏  11 MB/9.3 GB                  pulling manifest 
pulling a8cc1361f314:   0% ▕                  ▏  16 MB/9.3 GB                  pulling manifest 
pulling a8cc1361f314:   0% ▕                  ▏  20 MB/9.3 GB                  pulling manifest 
pulling a8cc1361f314:   0% ▕                  ▏  23 MB/9.3 GB                  pulling manifest 
pulling a8cc1361f314:   0% ▕                  ▏  28 MB/9.3 GB                  pulling manifest 
pulling a8cc1361f314:   0% ▕                  ▏  33 MB/9

KeyboardInterrupt: 

In [4]:
# === Krok 5: Weryfikacja ===

if ollama_running():
    r = requests.get('http://localhost:11434/api/tags', timeout=3)
    models = [m['name'] for m in r.json().get('models', [])]
    if models:
        print('Ollama działa. Dostępne modele:')
        for m in models:
            print(f'  - {m}')
        print('\nSetup zakończony — możesz otworzyć notebook z zajęć!')
    else:
        print('Ollama działa, ale brak pobranych modeli.')
        print(f'Uruchom w terminalu: ollama pull {model or "qwen3:8b"}')
else:
    print('Ollama nie odpowiada.')
    print('Na zajęciach skorzystaj z serwera prowadzącego (prowadzący poda adres IP).')

Ollama działa. Dostępne modele:
  - gpt-oss:20b
  - antoniprzybylik/llama-pllum:8b
  - deepseek-r1:32b
  - SpeakLeash/bielik-11b-v2.3-instruct:Q4_K_M
  - deepseek-r1:1.5b
  - llama3.2-vision:11b
  - deepseek-coder-v2:16b
  - mistral:7b
  - deepseek-r1:8b
  - deepseek-r1:14b
  - llama3.2:latest

Setup zakończony — możesz otworzyć notebook z zajęć!
